# **Importing Libraries**


In [131]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# **Loading the CSV file.**


In [132]:
df = pd.read_csv("movies.csv/movies.csv")

# _Data Cleaning Workflow for a Movies Dataset_


**1. Understand the Dataset First**


_Check rows & columns , Understand what each column means ,Identify target columns_


In [143]:
df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime
0,Blood Red Sky,(2021),"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0
1,Masters of the Universe: Revelation,(2021– ),"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0
2,The Walking Dead,(2010–2022),"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0
3,Rick and Morty,(2013– ),"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0
4,Army of Thieves,(2021),"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN


In [142]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   MOVIES    9999 non-null   object 
 1   YEAR      9355 non-null   object 
 2   GENRE     9919 non-null   object 
 3   RATING    8179 non-null   float64
 4   ONE-LINE  9999 non-null   object 
 5   STARS     9999 non-null   object 
 6   VOTES     8179 non-null   object 
 7   RunTime   7041 non-null   float64
dtypes: float64(2), object(6)
memory usage: 625.1+ KB


In [141]:
df.describe(include=["object", "int64", "float64"])

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime
count,9999,9355,9919,8179.000000,9999,9999,8179,7041.000000
unique,6817,438,510,NaN,8688,7877,4129,NaN
top,Bleach: Burîchi,(2020– ),\nComedy,NaN,\nAdd a Plot\n,\n,7,NaN
freq,65,892,852,NaN,1265,456,35,NaN
mean,NaN,NaN,NaN,6.921176,NaN,NaN,NaN,68.688539
std,NaN,NaN,NaN,1.220232,NaN,NaN,NaN,47.258056
min,NaN,NaN,NaN,1.100000,NaN,NaN,NaN,1.000000
25%,NaN,NaN,NaN,6.200000,NaN,NaN,NaN,36.000000
50%,NaN,NaN,NaN,7.100000,NaN,NaN,NaN,60.000000
75%,NaN,NaN,NaN,7.800000,NaN,NaN,NaN,95.000000


In [140]:
df.columns

Index(['MOVIES', 'YEAR', 'GENRE', 'RATING', 'ONE-LINE', 'STARS', 'VOTES',
       'RunTime'],
      dtype='object')

**2. Find Missing Values**


In [144]:
df.isnull().sum()

MOVIES         0
YEAR         644
GENRE         80
RATING      1820
ONE-LINE       0
STARS          0
VOTES       1820
RunTime     2958
dtype: int64

_You can see here **gross** column has so many null values so i'll drop this column_


In [138]:
df.drop(columns=["Gross"], inplace=True)

**i)Fix GENRE (Categorical/Text)**


_Use mode (most common value):Why? Bcoz genre is categorical data._


In [145]:
# The mode means the value that appears most frequently.
df["GENRE"].fillna(df["GENRE"].mode()[0], inplace=True)

**ii)Fix Numerical Columns**


_Use median instead of mean because movie data often has outliers._


In [148]:
df["RATING"].fillna(df["RATING"].median(), inplace=True)
df["VOTES"].fillna(df["VOTES"].median(), inplace=True)
df["RunTime"].fillna(df["RunTime"].median(), inplace=True)

**iii)YEAR Column**


_Before that first Extract only the valid 4-digit year._


In [159]:
# Step 1: Extract 4-digit years
df["YEAR"] = df["YEAR"].str.extract(r"(\d{4})")

# Step 2: Convert to numeric
df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce")

# Step 3: Fill missing values.
df["YEAR"].fillna(df["YEAR"].median(), inplace=True)

**iv) ONE-LINE column fix the add plots**


_Replace "add a plot" with NaN_


In [170]:
df['ONE-LINE'].fillna("Unknown" , inplace=True)

In [168]:
df["ONE-LINE"] = df["ONE-LINE"].replace("add a plot", np.nan)

_Remove rows with missing values in ONE-LINE_


In [172]:
df = df[df['ONE-LINE'] != "Unknown"]

**3)Remove Duplicates Data!**


_Movies datasets often contain duplicate movies._


In [150]:
df.duplicated().sum()

431

In [151]:
df.drop_duplicates(inplace=True)

**4. Fix Data Types**


_'VOTES' , 'RunTime' ideally should be integer, because vote & Runtime counts are whole numbers._


In [152]:
df["VOTES"] = df["VOTES"].fillna(df["VOTES"].median())
df["VOTES"] = df["VOTES"].astype(int)

df["RunTime"] = df["RunTime"].fillna(df["RunTime"].median())
df["RunTime"] = df["RunTime"].astype(int)

In [153]:
# Clean text columns
df["GENRE"] = df["GENRE"].str.replace("\n", "", regex=False)
df["ONE-LINE"] = df["ONE-LINE"].str.replace("\n", "", regex=False)
df["STARS"] = df["STARS"].str.replace("\n", "", regex=False)

**5. Standardize Text Data**


Movie datasets are messy with text formatting.

Common issues
"Action"
" action "
"ACTION"


In [154]:
# .title() :- "Action" , "Horror".
df["GENRE"] = df["GENRE"].str.strip().str.title()
df["MOVIES"] = df["MOVIES"].str.strip().str.title()
df["ONE-LINE"] = df["ONE-LINE"].str.strip().str.lower()
df["STARS"] = df["STARS"].str.strip().str.title()

Empty strings


In [155]:
(df["STARS"].str.strip() == "").sum()

388

In [156]:
df["STARS"] = df["STARS"].replace(r"^\s*$", np.nan, regex=True)

_Now replace NaN with 'Unknown'_


In [157]:
df["STARS"].fillna("Unknown", inplace=True)

**6.Validate the Dataset Again**

In [174]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8722 entries, 0 to 9979
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   MOVIES    8722 non-null   object 
 1   YEAR      8722 non-null   int32  
 2   GENRE     8722 non-null   object 
 3   RATING    8722 non-null   float64
 4   ONE-LINE  8722 non-null   object 
 5   STARS     8722 non-null   object 
 6   VOTES     8722 non-null   int32  
 7   RunTime   8722 non-null   int32  
dtypes: float64(1), int32(3), object(4)
memory usage: 511.1+ KB


In [176]:
df.isnull().sum()

MOVIES      0
YEAR        0
GENRE       0
RATING      0
ONE-LINE    0
STARS       0
VOTES       0
RunTime     0
dtype: int64

In [177]:
df.describe()

,YEAR,RATING,VOTES,RunTime
count,8722.000000,8722.000000,8.722000e+03,8722.000000
mean,2016.005389,6.934316,1.426297e+04,67.355538
std,7.391199,1.163907,6.792060e+04,42.199743
min,1932.000000,1.100000,5.000000e+00,1.000000
25%,2015.000000,6.300000,2.500000e+02,44.000000
50%,2018.000000,7.100000,7.890000e+02,60.000000
75%,2020.000000,7.700000,3.411750e+03,90.000000
max,2023.000000,9.900000,1.713028e+06,853.000000


**7.Save the Clean Dataset**

In [178]:
df.to_csv("Movies_cleaned.csv" , index=False)